In [1]:
import sklearn, imblearn
print(sklearn.__version__)
print(imblearn.__version__)


1.3.2
0.12.3


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import PowerTransformer, MinMaxScaler, OneHotEncoder, OrdinalEncoder
from imblearn.over_sampling import SMOTE

df = pd.read_csv('../output/transactions_data_completely_clean.csv')

# Feature definitions
num_cols_yeo = ['amount_src', 'amount_usd', 'fee', 'exchange_rate_src_to_dest', 
                'ip_risk_score', 'chargeback_history_count', 'risk_score_internal',
                'txn_velocity_1h', 'txn_velocity_24h', 'corridor_risk']

num_cols_minmax = ['account_age_days', 'device_trust_score']

ordinal_cols = ['kyc_tier']

nominal_cols = ['home_country', 'source_currency', 'dest_currency', 'channel', 'ip_country']

bin_cols = ["new_device", "location_mismatch"]

time_cols = ["hour_sin", "hour_cos"]


# Time features
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

df = df.drop(columns=['hour'])

categories_order = [['ENHANCED', 'STANDARD', 'LOW', 'UNKNOWN']]

preprocessor = ColumnTransformer(
    transformers=[
        ('yeo', PowerTransformer(method='yeo-johnson', standardize=True), num_cols_yeo),
        ('minmax', MinMaxScaler(), num_cols_minmax),
        ('ordinal', OrdinalEncoder(categories=categories_order), ordinal_cols),
        ('nominal', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), nominal_cols),
        ("bin", "passthrough", bin_cols),
        ("time", "passthrough", time_cols)
    ]
)

# Keep target column aside
y = df['is_fraud']
X = df.drop('is_fraud', axis=1)

# Fit-transform
X = preprocessor.fit_transform(df)

# One-hot column names
ohe_cols = preprocessor.named_transformers_['nominal'].get_feature_names_out(nominal_cols)

# Numeric + ordinal columns
other_cols = num_cols_yeo + num_cols_minmax + ordinal_cols

# Passthrough column names (already contains time_cols and bin_cols)
passthrough_cols = bin_cols + time_cols

# Full column names
all_cols = list(other_cols) + list(ohe_cols) + list(passthrough_cols)

# Create DataFrame
X_df = pd.DataFrame(X, columns=all_cols)

# Optionally, add target back for saving
X_df['is_fraud'] = y.values

# Save new df
os.makedirs('../output', exist_ok=True)
X_df.to_csv('../output/preprocessed_data.csv', index=False)

# Save preprocessor
os.makedirs('../artifacts', exist_ok=True)
joblib.dump(preprocessor, "../artifacts/preprocessor.pkl")

['../artifacts/preprocessor.pkl']

In [3]:
with pd.option_context('display.max_columns', None):
    display(X_df)

,amount_src,amount_usd,fee,exchange_rate_src_to_dest,ip_risk_score,chargeback_history_count,risk_score_internal,txn_velocity_1h,txn_velocity_24h,corridor_risk,account_age_days,device_trust_score,kyc_tier,home_country_CA,home_country_UK,home_country_UNKNOWN,home_country_US,source_currency_CAD,source_currency_GBP,source_currency_USD,dest_currency_CAD,dest_currency_CNY,dest_currency_EUR,dest_currency_GBP,dest_currency_INR,dest_currency_MXN,dest_currency_NGN,dest_currency_PHP,dest_currency_USD,channel_ATM,channel_MOBILE,channel_UNKNOWN,channel_WEB,ip_country_CA,ip_country_UK,ip_country_UNKNOWN,ip_country_US,new_device,location_mismatch,hour_sin,hour_cos,is_fraud
0,0.558773,0.543715,0.274827,-0.880988,-1.339161,-0.207593,-0.157747,-0.357508,-0.491316,-0.584829,0.239488,0.520121,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,-1.000000,-1.836970e-16,0
1,0.283074,-0.038214,0.271402,0.234591,0.948103,-0.207593,0.224151,-0.357508,1.874437,-0.584829,0.864717,0.472837,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,-0.866025,5.000000e-01,0
2,0.013657,0.002302,-0.441794,-0.048752,0.494635,-0.207593,-0.624739,-0.357508,-0.491316,-0.584829,0.334552,0.939638,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,-0.258819,9.659258e-01,0
3,-1.179389,-1.174986,-0.781820,-1.039431,1.021055,-0.207593,1.023780,-0.357508,-0.491316,-0.584829,0.133455,0.549296,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.258819,9.659258e-01,0
4,0.246329,0.233093,0.029904,0.979515,-1.356230,-0.207593,0.136044,-0.357508,-0.491316,-0.584829,0.234004,0.894366,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.707107,-7.071068e-01,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10645,0.535416,0.520465,0.565990,-0.880988,-0.099725,-0.207593,0.477221,2.731130,2.072090,-0.584829,0.677331,0.651911,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.500000,-8.660254e-01,0
10646,1.116897,0.857144,1.179623,1.554652,1.671107,4.817110,2.191430,2.804036,2.138217,-0.584829,0.007313,0.169014,2.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.965926,-2.588190e-01,1
10647,0.266931,-0.055425,0.185647,0.878613,1.837793,4.817110,1.877308,2.804036,2.129453,1.629408,0.018282,0.265594,2.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.000000,6.123234e-17,1
10648,-0.822315,-0.823616,-1.005458,0.862175,-0.147904,-0.207593,0.936262,-0.357508,2.072090,-0.584829,0.429616,0.772636,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.000000,1.000000e+00,0


In [4]:
X_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10650 entries, 0 to 10649
Data columns (total 42 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   amount_src                 10650 non-null  float64
 1   amount_usd                 10650 non-null  float64
 2   fee                        10650 non-null  float64
 3   exchange_rate_src_to_dest  10650 non-null  float64
 4   ip_risk_score              10650 non-null  float64
 5   chargeback_history_count   10650 non-null  float64
 6   risk_score_internal        10650 non-null  float64
 7   txn_velocity_1h            10650 non-null  float64
 8   txn_velocity_24h           10650 non-null  float64
 9   corridor_risk              10650 non-null  float64
 10  account_age_days           10650 non-null  float64
 11  device_trust_score         10650 non-null  float64
 12  kyc_tier                   10650 non-null  float64
 13  home_country_CA            10650 non-null  flo